# Phase 0 — Setup & Smoke Test
Thin driver: clones the repo, installs deps, runs the unit tests, then a tiny SDXL pilot (generate -> detector-score -> capture activations). All logic lives in `src/`; this notebook only drives it.

**Runtime:** GPU (T4 is enough).

In [ ]:
# 1. Clone repo + install deps (skip clone if already inside the repo)
import os
if not os.path.exists('src'):
    !git clone https://github.com/serinaqin/T2I-Count-Anomaly.git
    %cd T2I-Count-Anomaly
!pip install -q -r requirements.txt
!pip install -q pytest groundingdino-py

In [ ]:
# 2. Run the off-GPU unit tests (should all pass)
!pytest -q

In [ ]:
# 3. Imports
import sys; sys.path.insert(0, '.')
from src.prompts import generate_grid
from src.pipeline import (load_sdxl, generate,
                          catalog_attention_sites, ActivationCapture)
from src.detector import Detector, count_objects
from src.config import load_config
import pandas as pd, os

In [ ]:
# 4. Build the tiny pilot grid from the default config
cfg = load_config('configs/default.yaml')
grid = generate_grid(cfg.counts, cfg.objects, cfg.seeds)
print(len(grid), 'prompts in pilot')
grid[:3]

In [ ]:
# 5. Load SDXL and list the attention sites (the candidate probe catalog)
pipe = load_sdxl()
sites = catalog_attention_sites(pipe.unet)
print(len(sites), 'attention sites')
print(sites[:10])
# ---> paste this full list into docs/ARCHITECTURE.md (site catalog)
with open('results_sites.txt', 'w') as f:
    f.write('\n'.join(sites))

In [ ]:
# 6. Generate the pilot, score each image with the detector oracle
# (detector now applies non-max suppression to drop duplicate boxes)
from src.scoring import count_from_detections
det = Detector()
records = []  # (spec, image, detections, count) kept for eyeballing
rows = []
for p in grid:
    img = generate(pipe, p.text, p.seed, cfg.num_inference_steps)
    dets = det.detect(img, [p.obj])
    n = count_from_detections(dets, p.obj, cfg.score_threshold)
    records.append((p, img, dets, n))
    rows.append({'text': p.text, 'obj': p.obj, 'count': p.count,
                 'seed': p.seed, 'realized_count': n})
df = pd.DataFrame(rows)
os.makedirs('results', exist_ok=True)
df.to_csv('results/smoke_summary.csv', index=False)
df

### Eyeball the detector
Each image with the detector's boxes drawn on it, titled `prompt | asked N got M`. Check: do the red boxes land on distinct objects, and does M match what you see? This is how we judge whether the detector is trustworthy before scaling up.

In [ ]:
import matplotlib.pyplot as plt
from PIL import ImageDraw
n = len(records)
cols = 3
nrows = (n + cols - 1) // cols
fig, axes = plt.subplots(nrows, cols, figsize=(cols * 4, nrows * 4))
axes = axes.flatten()
for ax, (p, img, dets, cnt) in zip(axes, records):
    im = img.copy()
    draw = ImageDraw.Draw(im)
    for d in dets:
        if d['score'] >= cfg.score_threshold:
            draw.rectangle(d['box'], outline=(255, 0, 0), width=5)
    ax.imshow(im)
    ax.axis('off')
    mark = 'OK' if cnt == p.count else 'X'
    ax.set_title(f"{p.text} | asked {p.count} got {cnt}  {mark}",
                 fontsize=10)
for ax in axes[n:]:
    ax.axis('off')
plt.tight_layout()
plt.savefig('results/smoke_eyeball.png', dpi=90, bbox_inches='tight')
plt.show()

In [ ]:
# 7. Confirm activation hooks fire during generation (capture 3 sites, 2 steps)
with ActivationCapture(pipe.unet, sites[:3]) as cap:
    _ = generate(pipe, grid[0].text, grid[0].seed, num_inference_steps=2)
print({k: tuple(v.shape) for k, v in cap.acts.items()})

## (Optional) Re-score previously generated images from Google Drive
Re-scores your prior single-object images with the **detector oracle** (vs the old VLM scorer) — no regeneration needed. Set `USE_DRIVE = True` to run. Expects images at `MyDrive/ColabNotebooks/T2I-Count-Anomaly/generated-images-single-count/` and a manifest CSV (with an object/label column) in the same base folder.

In [ ]:
USE_DRIVE = False  # flip to True to re-score prior Drive images

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    from pathlib import Path
    from PIL import Image
    import glob
    base = Path('/content/drive/MyDrive/ColabNotebooks/T2I-Count-Anomaly')
    img_dir = base / 'generated-images-single-count'
    # find a manifest CSV that maps image index -> object label
    manifest = None
    for cand in list(base.glob('*.csv')) + list(img_dir.glob('*.csv')):
        try:
            m = pd.read_csv(cand)
        except Exception:
            continue
        if any(c in m.columns for c in ['obj', 'object', 'prompt']):
            manifest = m; print('using manifest', cand.name); break
    if manifest is None:
        print('No manifest with an obj/object/prompt column found under', base)
        print('Available CSVs:', [p.name for p in base.glob("*.csv")])
    else:
        det = det if 'det' in dir() else Detector()
        def label_for(row):
            for c in ['obj', 'object']:
                if c in row and isinstance(row[c], str):
                    return row[c]
            # fall back: last word of the prompt, singularized crudely
            w = str(row.get('prompt', '')).split()[-1]
            return w[:-1] if w.endswith('s') else w
        out = []
        for i, row in manifest.iterrows():
            p = img_dir / f'{i:05d}.png'
            if not p.exists():
                continue
            obj = label_for(row)
            n = count_objects(det, Image.open(p).convert('RGB'), obj,
                              cfg.score_threshold)
            rec = {'idx': i, 'obj': obj, 'detector_count': n}
            for c in ['count', 'prompt', 'seed']:
                if c in row: rec[c] = row[c]
            out.append(rec)
        rescored = pd.DataFrame(out)
        rescored.to_csv('results/drive_rescored.csv', index=False)
        print('re-scored', len(rescored), 'prior images with the detector')
        rescored.head()